# Lab 20-06 · Measuring a Converter

**Predict first**, then run. The handout is [README.md](README.md). The converter is a behavioral 12-bit SAR ADC with a 1.6 V range.

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import adc

N, VFS = 12, 1.6
LSB = VFS / 2 ** N


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}


def ideal(v):
    return np.clip(np.floor(v / LSB + 0.5), 0, 2 ** N - 1) * LSB

## Part 1 · Quantization spectra

*Predict* the SFDR at bin 1001 and at bin 64 of a 4096-point record.

In [2]:
n = 4096
t = np.arange(n)
for m in (1001, 64):
    r = adc.spectrum_metrics(ideal(0.8 + 0.8 * (1 - 1e-9) * np.sin(2 * np.pi * m * t / n)), m)
    print(f"bin {m:4d}: SINAD {r['sinad']:.2f} dB, SFDR {r['sfdr']:.1f} dBc")
rng = np.random.default_rng(20061)
r = adc.spectrum_metrics(ideal(0.8 + 0.784 * np.sin(2 * np.pi * 64 * t / n) + rng.normal(0, 0.5 * LSB, n)), 64)
print(f"bin   64 with dither: SINAD {r['sinad']:.2f} dB, SFDR {r['sfdr']:.1f} dBc")

bin 1001: SINAD 73.65 dB, SFDR 93.7 dBc
bin   64: SINAD 73.83 dB, SFDR 79.9 dBc
bin   64 with dither: SINAD 67.72 dB, SFDR 92.4 dBc


## Part 2 · The flash ladder

*Predict* the seven taps of Example 20-06.2.

In [3]:
print(ngspice("ladder.cir"))
R = np.array([51, 98, 103, 100, 97, 104, 100, 150.0])
taps = 0.8 * np.cumsum(R)[:7] / R.sum()
offs = np.array([15, -20, 35, 0, -10, 40, -5]) * 1e-3
for name, tt in (("ladder only", taps), ("with comparator offsets", taps + offs)):
    d, i = adc.dnl_inl(tt)
    print(f"{name:24s}: max |DNL| {np.abs(d).max():.3f} LSB, max |INL| {np.abs(i).max():.3f} LSB")

{'t1': 0.05080946, 't2': 0.1484433, 't3': 0.2510585, 't4': 0.3506849, 't5': 0.4473225, 't6': 0.550934, 't7': 0.6505604, 'dnlmax': 0.03654485}
ladder only             : max |DNL| 0.037 LSB, max |INL| 0.033 LSB
with comparator offsets : max |DNL| 0.631 LSB, max |INL| 0.435 LSB


## Part 3 · SAR mismatch

*Predict* where the largest DNL of a binary-weighted SAR with 0.5% unit mismatch occurs.

In [4]:
w = adc.sar_weights(N, 0.005, np.random.default_rng(2006))
T = adc.sar_transitions(w, VFS)
dnl, inl = adc.dnl_inl(T)
print(f"max |DNL| {np.abs(dnl).max():.3f} LSB at code {np.argmax(np.abs(dnl)) + 1}; max |INL| {np.abs(inl).max():.3f} LSB")
fig, (a1, a2) = plt.subplots(2, 1, sharex=True)
a1.plot(np.arange(1, 4095), dnl, lw=0.5); a1.set_ylabel("DNL (LSB)")
a2.plot(np.arange(1, 4096), inl, lw=0.8); a2.set_ylabel("INL (LSB)"); a2.set_xlabel("code"); plt.show()

max |DNL| 0.370 LSB at code 2047; max |INL| 0.328 LSB


## Part 4 · Histogram test

*Predict* the INL error with random phases and with coherent phases for 2^22 samples.

In [5]:
ns, A = 2 ** 22, 1.02 * 0.8
coherent = 2 * np.pi * (np.arange(ns) * 1048573 / ns + 0.123)
random_ = np.random.default_rng(2007).uniform(0, 2 * np.pi, ns)
for name, ph in (("coherent", coherent), ("random", random_)):
    codes = adc.sar_code(0.8 + A * np.sin(ph), w, VFS)
    th = adc.histogram_transitions(codes, 2 ** N, A, 0.8)
    dh, ih = adc.dnl_inl(th)
    print(f"{name:9s}: DNL error max {np.abs(dh - dnl).max():.4f} (rms {np.sqrt(np.mean((dh - dnl) ** 2)):.4f}) LSB, "
          f"INL error max {np.abs(ih - inl).max():.4f} LSB")

coherent : DNL error max 0.0024 (rms 0.0007) LSB, INL error max 0.0013 LSB


random   : DNL error max 0.1278 (rms 0.0355) LSB, INL error max 1.4349 LSB


## Part 5 · FFT test and figures of merit

*Predict* the SINAD and ENOB from 20-05's SNR (68.8 dB) and HD3 (−64.4 dBc).

In [6]:
n = 16384
f, m = adc.coherent_frequency(100e6, n, 49e6)
sig_th = math.sqrt(206.2791e-6 ** 2 - (LSB / math.sqrt(12)) ** 2)
a3 = 4 * 10 ** (-64.40879 / 20) / 0.8 ** 2
vd = 0.8 * (1 - 1e-6) * 0.999 * np.sin(2 * np.pi * m * np.arange(n) / n)
vin = 0.8 + vd - a3 * vd ** 3 + np.random.default_rng(2008).normal(0, sig_th, n)
x = adc.sar_code(vin, w, VFS) * LSB
r = adc.spectrum_metrics(x, m, full_scale_amp=0.8)
print(f"f_in = {f / 1e6:.4f} MHz: SNR {r['snr']:.2f} dB, THD {r['thd']:.2f} dBc, SINAD {r['sinad']:.2f} dB, "
      f"SFDR {r['sfdr']:.2f} dBc, ENOB(FS) {r['enob_fs']:.2f}")
sinad_fs = r["sinad"] + 20 * math.log10(0.8 / r["amp"])
print(f"FoM_W = {adc.fom_walden(1.5e-3, 100e6, r['enob_fs']) * 1e15:.1f} fJ/step, "
      f"FoM_S = {adc.fom_schreier(sinad_fs, 50e6, 1.5e-3):.1f} dB (thermal limit {adc.thermal_limit_fom_s():.1f} dB)")
s = np.abs(np.fft.rfft(x - x.mean())) ** 2 / n ** 2 * 2
plt.figure(); plt.plot(np.arange(s.size) * 100e6 / n / 1e6, 10 * np.log10(s / (0.8 ** 2 / 2) + 1e-20), lw=0.5)
plt.ylim(-160, 5); plt.xlabel("frequency (MHz)"); plt.ylabel("dBFS"); plt.show()

f_in = 49.0051 MHz: SNR 68.67 dB, THD -63.96 dBc, SINAD 62.69 dB, SFDR 63.99 dBc, ENOB(FS) 10.13
FoM_W = 13.4 fJ/step, FoM_S = 167.9 dB (thermal limit 191.8 dB)
